In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd
from tqdm import tqdm
import logging
import re
import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


class DistilBertForSequenceClassificationWithRating(nn.Module):
    def __init__(self, num_labels=6):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.pre_classifier = nn.Linear(self.distilbert.config.dim + 1, self.distilbert.config.dim)
        self.classifier = nn.Linear(self.distilbert.config.dim, num_labels)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask=None, rating=None):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs[0]
        pooled_output = hidden_state[:, 0]  # CLS token output

        if rating is not None:
            rating = rating.unsqueeze(-1)
            pooled_output = torch.cat((pooled_output, rating), dim=1)

        pooled_output = self.pre_classifier(pooled_output)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits


class TextDataset(Dataset):
    def __init__(self, texts, ratings, labels, tokenizer, max_length=160):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length, return_tensors="pt")
        self.ratings = torch.tensor(ratings, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        item = {key: self.encodings[key][idx].clone().detach() for key in self.encodings}
        item['ratings'] = self.ratings[idx].clone().detach()
        item['labels'] = self.labels[idx].clone().detach()
        return item

    def __len__(self):
        return len(self.labels)


def clean_text(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r"[^a-zA-Z0-9\s']", '', text)
    return text


def map_ratings_to_labels(rating):
    if rating in [1, 2]:
        return 'extremely negative'
    elif rating in [3, 4]:
        return 'negative'
    elif rating == 5:
        return 'slightly negative'
    elif rating == 6:
        return 'slightly positive'
    elif rating in [7, 8]:
        return 'positive'
    else:
        return 'extremely positive'


def load_and_balance_data(filepath, tokenizer, test_size=0.1, max_samples_per_class=None):
    df = pd.read_csv(filepath, encoding='ISO-8859-1')
    df['rating_category'] = df['rating'].apply(map_ratings_to_labels)
    df['combined_text'] = df['rating_category'] + ": " + df['review'].astype(str)
    df['combined_text'] = df['combined_text'].apply(clean_text)

    categories = ['extremely negative', 'negative', 'slightly negative', 'slightly positive', 'positive', 'extremely positive']
    dfs = [df[df['rating_category'] == category] for category in categories]

    if max_samples_per_class is None:
        max_samples_per_class = min(len(df_cat) for df_cat in dfs)

    dfs_resampled = [df_cat.sample(n=max_samples_per_class, random_state=42) for df_cat in dfs]
    df_balanced = pd.concat(dfs_resampled)

    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
    label_mapping = {category: i for i, category in enumerate(categories)}
    labels = df_balanced['rating_category'].map(label_mapping).values
    texts = df_balanced['combined_text'].tolist()
    ratings = df_balanced['rating'].values

    train_texts, test_texts, train_labels, test_labels, train_ratings, test_ratings = train_test_split(
        texts, labels, ratings, test_size=test_size, random_state=42, stratify=labels)

    return train_texts, train_ratings, train_labels, test_texts, test_ratings, test_labels


def compute_weights(labels):
    class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    return torch.tensor(class_weights, dtype=torch.float)


def train_model(model, train_loader, class_weights, device='cpu', lr=1e-5, epochs=1):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(device))
    for epoch in range(epochs):
        model.train()
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ratings = batch['ratings'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask, ratings)
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()
        logging.info(f"Epoch {epoch+1} completed. Loss: {loss.item()}")
    logging.info("Training complete.")
    torch.save(model, 'distilbert_classification_model_with_6_sentiments.pth')
    torch.save(model.state_dict(), 'distilbert_classification_model_state_dict_with_6_sentiments.pth')
    logging.info("Model and state dictionary have been saved.")


def evaluate_model(model, data_loader, device='cpu'):
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ratings = batch['ratings'].to(device)
            labels = batch['labels'].to(device)
            logits = model(input_ids, attention_mask, ratings)
            preds = torch.argmax(logits, dim=-1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted')
    accuracy = accuracy_score(true_labels, predictions)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }


def main():
    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    model = DistilBertForSequenceClassificationWithRating(num_labels=6)
    train_texts, train_ratings, train_labels, test_texts, test_ratings, test_labels = load_and_balance_data(
        r"C:\Users\jpers\Desktop\NLP\drugsComTrain_raw.csv", tokenizer, max_samples_per_class=)
    class_weights = compute_weights(train_labels)
    train_dataset = TextDataset(train_texts, train_ratings, train_labels, tokenizer)
    test_dataset = TextDataset(test_texts, test_ratings, test_labels, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    train_model(model, train_loader, class_weights)
    performance = evaluate_model(model, test_loader)
    print(performance)


if __name__ == "__main__":
    main()
